In [1]:
import pandas as pd
import os

In [2]:
bucket_name = "pair-email-classification"
rvz_mit_gvz_object_base = "data/aftercourt/gvz_mit_gvz/"
rvz_mit_gvz_local_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/rvz_mit_gvz"

In [3]:
rvz_mit_gvz_local_paths = [os.path.join(rvz_mit_gvz_local_dir, filename) for filename in os.listdir(rvz_mit_gvz_local_dir) if filename.endswith(".pdf")]

In [4]:
print(f"Found {len(rvz_mit_gvz_local_paths)} PDFs") 

Found 53 PDFs


In [5]:
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from utils.use_textract_utils import parse_local_pdfs_with_textract

In [ ]:
rvz_mit_gvz_texts = parse_local_pdfs_with_textract(
    local_pdf_paths=rvz_mit_gvz_local_paths,
    s3_object_key_base=rvz_mit_gvz_object_base,
    s3_bucket_name=bucket_name,
    use_page_markers=False,
)

In [6]:
output_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/classification/rvz_mit_gvz/dataset"

In [ ]:

rows = [
    {"s3_key": k, "text": v} for k, v in rvz_mit_gvz_texts.items()
] 

df = pd.DataFrame(rows)
df.to_parquet(os.path.join(output_dir, "rvz_mit_gvz_objectkey_to_texts.parquet"), index=False)

In [ ]:
df

In [ ]:
print(df.iloc[4]["text"])

# add new data to final_raw_dataset.csv

In [7]:
raw_data = pd.read_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv')
raw_data.columns

Index(['ticket_uuid', 'attachment_id', 'text', 'object_key', 'document_type',
       'cleaned_text', 'data', 'is_pfub', 'is_ladung', 's3_link',
       'textract_job_id', 'textract_s3_link', 'is_ve_with_invoice',
       'text_w_pages', 'is_da_with_invoice', 'is_va'],
      dtype='object')

In [8]:
# we already have some rvz mit gvz data called bailiff_ip, change the label name of them

raw_data['document_type'] = raw_data['document_type'].replace({'bailiff_ip': 'rvz_mit_gvz'})

In [9]:
raw_data['document_type'] .value_counts()

document_type
ladung_va                                 2030
mail_attachments                           984
monierung_mb                               545
attachment_and_transfer_order              535
approved_seizure                           523
court_inbox                                346
fp_protocol                                220
fp_invoice                                 106
vermögensverzeichnis                        75
drittauskunft                               71
approved_attachment_and_transfer_order      64
enforcement_order                           23
tbd                                         22
rvz_mit_gvz                                 20
va_dritt_invoice_protokol_combination       18
contradiction                                3
neg_drittauskunft_hard                       1
Name: count, dtype: int64

In [ ]:
import re
df.rename(columns={"s3_key": "object_key"}, inplace=True)
df['document_type'] = 'rvz_mit_gvz'
df['is_ladung'] = False
df['is_pfub'] = False

In [ ]:
# df.to_csv(os.path.join(output_dir, "add_data_rvz_mit_gvz.csv"), index=False)

In [10]:
df = pd.read_csv(os.path.join(output_dir, "add_data_rvz_mit_gvz.csv"))

In [11]:
df

,object_key,text,document_type,is_ladung,is_pfub
0,data/aftercourt/gvz_mit_gvz/Dokument_66225_311...,Michael Gerlin\nSchillerstraße 9\nGerichtsvoll...,rvz_mit_gvz,False,False
1,data/aftercourt/gvz_mit_gvz/0_sammel120260113-...,Obergerichtsvollzieher b. AG Marburg\n35274 Ki...,rvz_mit_gvz,False,False
2,data/aftercourt/gvz_mit_gvz/Dokument_55426_160...,Gerichtsvollzieher Neumaier\nBüroanschrift:\nA...,rvz_mit_gvz,False,False
3,data/aftercourt/gvz_mit_gvz/96dca3ee-d92e-42e3...,Simone Bleyl\nSegelfliegerdamm 89\nGerichtsvol...,rvz_mit_gvz,False,False
4,data/aftercourt/gvz_mit_gvz/Dokument_67926_300...,Rainer Jagemann\nErlenweg 13\nObergerichtsvoll...,rvz_mit_gvz,False,False
5,data/aftercourt/gvz_mit_gvz/0_Sammel120260313-...,Gerichtsvollzieher\nIndustriestraße 20\nE-Mail...,rvz_mit_gvz,False,False
6,data/aftercourt/gvz_mit_gvz/Dokument_40526_160...,"Laura Soth-Igelmann\nAmtsgericht Bersenbrück, ...",rvz_mit_gvz,False,False
7,data/aftercourt/gvz_mit_gvz/combined_pdf_DR-II...,Pestalozzistraße 6\n02826 Görlitz\nTel. 03581/...,rvz_mit_gvz,False,False
8,data/aftercourt/gvz_mit_gvz/1_Sammel120260623-...,Gerichtsvollzieher Schlegel\nKlosterstraße 14 ...,rvz_mit_gvz,False,False
9,data/aftercourt/gvz_mit_gvz/Dokument_9726_2502...,Jörg Ehlert\nKirchstraße 13-15\nGerichtsvollzi...,rvz_mit_gvz,False,False


add new data using update raw data script

In [12]:
from src.data_handling.update_raw_data import update_raw_data

updated_raw_data = update_raw_data(raw_data, df)
print(updated_raw_data.shape)
updated_raw_data.tail()

Starting raw data update. Raw data: 5586 rows, New data: 53 rows.
Validating required columns: ['text', 'document_type', 'is_pfub', 'is_ladung']
All required columns validated successfully.
Filling generated columns for 53 rows.
Validating dtypes between raw data and new data.
These ticket uuids are duplicated: 
0     798a2aa9-9ea5-57fb-a3b5-c17c95edd012
1     61adde78-379c-5921-a27a-8201dd6f375d
10    409a01d8-7b4d-571d-ba49-09e62cbf4256
14    fe3e4cf4-85ce-5c1b-bb46-90e04711f574
16    14c734e0-8292-5f9a-b47e-5ea67f5f8585
25    2d8fabf4-998d-5c5a-9eef-d4588b40651a
29    872dfe0e-a7af-5aba-914e-86b569a217cc
33    7710267b-82ca-5a8c-ac2c-84ef47d0d94f
35    d2925bff-b374-5eca-afe1-238bb7733b81
39    4bb586db-a0cd-5cc3-8b02-284a4502e7c3
40    d4b6afba-fa59-5a89-b91a-572da03e842c
41    6e515eec-7ada-5900-99bc-4d7b9f49e205
43    eac61279-9ac2-5d7c-8b1f-c2acb4ab6df3
52    ac7d996f-1482-5744-95fe-0823ef371cbb
Name: ticket_uuid, dtype: object
Raw data update complete. Total rows: 5625 (added 3

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va
5620,f99f2a28-9f9c-524c-82cb-47236b46df35,5a16e79d-ef76-52a8-8b8d-a2081de51efa,Gerichtsvollzieher\nBischof-Fischer-Str. 86\nA...,data/aftercourt/gvz_mit_gvz/0b03532c-272c-4177...,rvz_mit_gvz,NaN,NaN,False,False,NaN,bd41ea3911d343a5e5e5faf0d38d2d1b4e7cb3f79dcd12...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,NaN
5621,cdd54c7c-d2f8-533a-a7e8-368d60ceff7f,600aa8c2-6faa-5360-b079-04393f900e74,Hauptgerichtsvollzieher\nGmelchstraße 31\nFran...,data/aftercourt/gvz_mit_gvz/0c3e0762-c5e2-48ff...,rvz_mit_gvz,NaN,NaN,False,False,NaN,e2351ba20f9e05d943bf5bd0cd9704da84cd52715cff60...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,NaN
5622,f8b6754c-5e4b-511d-bffc-a9b39f71801a,6f47bc91-8ff4-54b4-8283-a0feb8d3f1a2,M. Graetz\nHellersdorfer Weg 35\nObergerichtsv...,data/aftercourt/gvz_mit_gvz/Dokument_32626_230...,rvz_mit_gvz,NaN,NaN,False,False,NaN,cb36a9e439a04f4bd6f5aa1f12ea612139cc6b399e923c...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,NaN
5623,e680a226-3d52-576d-9543-2520342e8108,d4a72d07-4a8c-5bea-9506-8b14856f9f4b,Obergerichtsvollzieher Römer\nAmtsgericht Kais...,data/aftercourt/gvz_mit_gvz/636206df-aaeb-4126...,rvz_mit_gvz,NaN,NaN,False,False,NaN,4bdbba375d69c33c66d05ec499e337e38def33ba9d3784...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,NaN
5624,3c4b096e-d261-5cfb-a78f-06ec5af76ca7,7c5d78a8-24b6-58d5-98ba-cfa1a1e89c6d,Lars Schimming\nAstfelder Straße 8\nGerichtsvo...,data/aftercourt/gvz_mit_gvz/Dokument_14026_010...,rvz_mit_gvz,NaN,NaN,False,False,NaN,256a0e7d104d01d3e66cf36b150d5ec7d343e9d7555488...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,NaN


we have some data already existing, check what are they and theirs labels

In [13]:
already_existing_uuids = [
    "798a2aa9-9ea5-57fb-a3b5-c17c95edd012",
    "61adde78-379c-5921-a27a-8201dd6f375d",
    "409a01d8-7b4d-571d-ba49-09e62cbf4256",
    "fe3e4cf4-85ce-5c1b-bb46-90e04711f574",
    "14c734e0-8292-5f9a-b47e-5ea67f5f8585",
    "2d8fabf4-998d-5c5a-9eef-d4588b40651a",
    "872dfe0e-a7af-5aba-914e-86b569a217cc",
    "7710267b-82ca-5a8c-ac2c-84ef47d0d94f",
    "d2925bff-b374-5eca-afe1-238bb7733b81",
    "4bb586db-a0cd-5cc3-8b02-284a4502e7c3",
    "d4b6afba-fa59-5a89-b91a-572da03e842c",
    "6e515eec-7ada-5900-99bc-4d7b9f49e205",
    "eac61279-9ac2-5d7c-8b1f-c2acb4ab6df3",
    "ac7d996f-1482-5744-95fe-0823ef371cbb",
]

In [14]:
check = raw_data[raw_data['ticket_uuid'].isin(already_existing_uuids)]

In [15]:
check

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va
5260,798a2aa9-9ea5-57fb-a3b5-c17c95edd012,49a46bf9-1273-5bad-96e0-1f6084a3cf9f,Michael Gerlin\nSchillerstraße 9\nGerichtsvoll...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,michael gerlin\nschillerstraße 9\ngerichtsvoll...,NaN,False,False,NaN,622143c54f400b40efb8c56be4e5bf79cfe72f4eec4bc6...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5261,61adde78-379c-5921-a27a-8201dd6f375d,d0daa6ac-42a7-58e7-829d-51126df14f6e,Obergerichtsvollzieher b. AG Marburg\n35274 Ki...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,obergerichtsvollzieher b. ag marburg\n35274 ki...,NaN,False,False,NaN,36801d468c7366407ef4a403a791dad8ac512a85109d2b...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5262,409a01d8-7b4d-571d-ba49-09e62cbf4256,1769e0c8-c3e7-53a2-86bc-f8d338c2ccc8,Amtsgericht Hamburg-St.Georg\nGerichtsvollzieh...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,amtsgericht hamburg-st.georg\ngerichtsvollzieh...,NaN,False,False,NaN,b2ffd24f9deece7fe70968c998777e69f12c0e3122e9f9...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5264,fe3e4cf4-85ce-5c1b-bb46-90e04711f574,fefde67e-9396-5234-b2ca-997663b5fda0,Obergerichtsvollzieherin\n45665 Recklinghausen...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,obergerichtsvollzieherin\n45665 recklinghausen...,NaN,False,False,NaN,c64fad6f107438c7df3b17f44b1e8942eb6ad02e1cb413...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5265,14c734e0-8292-5f9a-b47e-5ea67f5f8585,7f3c58e2-b26c-5b6a-84f8-c37a5d1fd050,Yvonne Lampe\nUthleber Str. 24 (Scheunenhof)\n...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,yvonne lampe\nuthleber str. 24 (scheunenhof)\n...,NaN,False,False,NaN,b3baf10d84a04a208c4d0410326fe28598dcdc1b510f8d...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5266,2d8fabf4-998d-5c5a-9eef-d4588b40651a,48873a6f-5f41-5df6-9436-aaad10c4b67d,Obergerichtsvollzieherin\nWestliche Ringstraße...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,obergerichtsvollzieherin\nwestliche ringstraße...,NaN,False,False,NaN,e58f06f4985ce257223460b109c81822b92e94d218109d...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5267,872dfe0e-a7af-5aba-914e-86b569a217cc,01ebcad0-eb02-5abc-aeb3-98ac539c5a3d,Gerichtsvollzieher\nAmtsgericht\nM. Götz\nSonn...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,gerichtsvollzieher\namtsgericht\nm. götz\nsonn...,NaN,False,False,NaN,f9668450387e41e3332a483b6ac749c9d63e76f8c2c747...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5269,7710267b-82ca-5a8c-ac2c-84ef47d0d94f,0608598d-df71-595a-a129-21330f705739,Obergerichtsvollzieher\nBüroanschrift\nMartin ...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,obergerichtsvollzieher\nbüroanschrift\nmartin ...,NaN,False,False,NaN,4e1f79ae646d830feca03e5191eea9682fbcd4b433164d...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5270,d2925bff-b374-5eca-afe1-238bb7733b81,ce1f74a1-0a49-57da-b4c6-f394eb68209b,Obergerichtsvollzieherin\nBramfelder Straße 10...,data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,obergerichtsvollzieherin\nbramfelder straße 10...,NaN,False,False,NaN,4d4220c951b6ba02a11dfab53ab10356b3d5a3057f8777...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
5272,4bb586db-a0cd-5cc3-8b02-284a4502e7c3,c61fb0c7-3bc9-584e-a9e6-4ce24b0ecf13,"Myriam Wehmeyer\nc/o Amtsgericht Rotenburg,Am\...",data/aftercourt/ladung_fp_instplan_with_bailif...,rvz_mit_gvz,"myriam wehmeyer\nc/o amtsgericht rotenburg,am\...",NaN,False,False,NaN,ea578e5a3589fc0c5b640d644d57e917f384c0458f08ef...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False


In [16]:
updated_raw_data.document_type.value_counts()

document_type
ladung_va                                 2030
mail_attachments                           984
monierung_mb                               545
attachment_and_transfer_order              535
approved_seizure                           523
court_inbox                                346
fp_protocol                                220
fp_invoice                                 106
vermögensverzeichnis                        75
drittauskunft                               71
approved_attachment_and_transfer_order      64
rvz_mit_gvz                                 59
enforcement_order                           23
tbd                                         22
va_dritt_invoice_protokol_combination       18
contradiction                                3
neg_drittauskunft_hard                       1
Name: count, dtype: int64

when sure, overwrite the new raw data and add it with dvc add

In [17]:
# final data looks good, save it to csv (overwrite the existing one)
updated_raw_data.to_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv', index=False)